# Orbit: Qwen2.5-1.5B QLoRA on a free Colab T4

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shmorsi/orbit-multi-agent/blob/main/finetune/colab_train.ipynb)

This notebook trains `finetune/configs/qlora_medium.json` only: Qwen2.5-1.5B-Instruct, QLoRA. It targets one free T4 sitting for training plus the three frozen evals. It does not use the 0.5B config.

Run the cells from top to bottom. Upload a project zip to `/content/orbit.zip` to train that copy. With no zip, the next cell clones `https://github.com/shmorsi/orbit-multi-agent` (override with the `REPO_URL` environment variable). The install cell uninstalls torchao, because Colab's preinstalled build makes `PeftModel.from_pretrained` raise ImportError. It must not replace Colab's CUDA torch build.

Iterate on your own machine with `python -m orbit_agents.eval.run --split tune`. This notebook does not run that split.

Do not score the frozen test in a loop. Training writes an adapter and a loss curve, then the frozen test is scored once.

The agent harness is v3. Frozen-test scores from this notebook are written under `results/v3_qwen1.5b/`. A file that is already measured is skipped. The pending files at the top of `results/` are not replaced. Set `ORBIT_RESCORE=1` and run the rescore cell to fill any of those four files that are still missing. That cell does not train.


In [ ]:
import os
if os.environ.get('ORBIT_MOUNT_DRIVE') == '1':
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted')
else:
    print('Drive mount skipped. Set ORBIT_MOUNT_DRIVE=1 and rerun this cell to mount. Copies still go to /content/orbit-checkpoints.')


In [ ]:
import os, shutil, subprocess, zipfile
from pathlib import Path

zip_path = Path('/content/orbit.zip')
if zip_path.exists():
    dest = Path('/content/orbit')
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(dest)
    if not (dest / 'pyproject.toml').exists():
        children = [path for path in dest.iterdir() if path.is_dir() and (path / 'pyproject.toml').exists()]
        if len(children) != 1:
            raise SystemExit('The zip does not contain pyproject.toml at its top level.')
        os.chdir(children[0])
    else:
        os.chdir(dest)
    print('Using the uploaded project.')
else:
    print('No /content/orbit.zip. The next cell clones the default repo.')


In [ ]:
import json, os, shutil, subprocess
from pathlib import Path

repo = os.environ.get('REPO_URL', 'https://github.com/shmorsi/orbit-multi-agent').strip()
if Path('pyproject.toml').exists():
    print('Project already present. Skipping clone.')
else:
    target = Path('/content/orbit-repo')
    if target.exists():
        shutil.rmtree(target)
    subprocess.check_call(['git', 'clone', repo, str(target)])
    os.chdir(target)
    print('cloned', repo)
print(subprocess.check_output(['nvidia-smi'], text=True))
config = json.loads(Path('finetune/configs/qlora_medium.json').read_text())
assert config['base_model'] == 'Qwen/Qwen2.5-1.5B-Instruct'
assert config['method'] == 'qlora'
print('project', Path.cwd())
print('base model', config['base_model'])


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade-strategy', 'only-if-needed', '-r', 'finetune/requirements-colab.txt'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'])
# Subprocess import: Colab's kernel puts /content on sys.path and can load the unzipped folder as a namespace package.
print(subprocess.check_output([sys.executable, '-c', 'import orbit_agents, os; print(orbit_agents.__version__, os.path.dirname(orbit_agents.__file__))'], text=True))
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
subprocess.check_call([sys.executable, 'finetune/leakage.py'], env=env)

def save_step(paths):
    destinations = [Path('/content/orbit-checkpoints')]
    if Path('/content/drive/MyDrive').exists():
        destinations.append(Path('/content/drive/MyDrive/orbit-v2'))
    extra = os.environ.get('ORBIT_SAVE_DIR', '').strip()
    if extra:
        destinations.append(Path(extra))
    for dest in destinations:
        dest.mkdir(parents=True, exist_ok=True)
        for raw in paths:
            path = Path(raw)
            if not path.exists():
                raise SystemExit(f'missing {path}')
            target = dest / path.name
            if path.is_dir():
                if target.exists():
                    shutil.rmtree(target)
                shutil.copytree(path, target)
            else:
                shutil.copy2(path, target)
            print('saved', target)


In [ ]:
import os, subprocess, sys
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
command = [sys.executable, 'finetune/train.py', '--config', 'finetune/configs/qlora_medium.json', '--output', 'finetune/outputs']
print(' '.join(command), flush=True)
subprocess.check_call(command, env=env)
save_step([
    'finetune/outputs/adapter',
    'finetune/outputs/loss_curve.png',
    'finetune/outputs/RUN.json',
    'finetune/outputs/train_log.jsonl',
])


In [ ]:
from pathlib import Path
path = Path('finetune/outputs/loss_curve.png')
if not path.exists():
    raise SystemExit('Training did not write a loss curve. There is no score to invent.')
from IPython.display import Image, display
display(Image(filename=str(path)))
print(Path('finetune/outputs/RUN.json').read_text())


In [ ]:
import json, os, subprocess, sys
from pathlib import Path
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
dest = Path('results/v3_qwen1.5b/single_base.json')
dest.parent.mkdir(parents=True, exist_ok=True)
already = False
if dest.is_file():
    payload = json.loads(dest.read_text())
    already = payload.get('status') == 'measured' and payload.get('kind') == 'model_eval'
if already:
    print(f'{dest} is already measured. Not scoring the frozen test again.')
else:
    command = [
        sys.executable, '-m', 'orbit_agents.eval.run',
        '--split', 'test',
        '--mode', 'single',
        '--backend', 'transformers',
        '--model', 'Qwen/Qwen2.5-1.5B-Instruct',
        '--out', str(dest),
    ]
    print(' '.join(command), flush=True)
    subprocess.check_call(command, env=env)
    save_step([str(dest)])


In [ ]:
import json, os, subprocess, sys
from pathlib import Path
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
dest = Path('results/v3_qwen1.5b/team_base.json')
dest.parent.mkdir(parents=True, exist_ok=True)
already = False
if dest.is_file():
    payload = json.loads(dest.read_text())
    already = payload.get('status') == 'measured' and payload.get('kind') == 'model_eval'
if already:
    print(f'{dest} is already measured. Not scoring the frozen test again.')
else:
    command = [
        sys.executable, '-m', 'orbit_agents.eval.run',
        '--split', 'test',
        '--mode', 'team',
        '--backend', 'transformers',
        '--model', 'Qwen/Qwen2.5-1.5B-Instruct',
        '--out', str(dest),
    ]
    print(' '.join(command), flush=True)
    subprocess.check_call(command, env=env)
    save_step([str(dest)])


In [ ]:
import json, os, subprocess, sys
from pathlib import Path
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
dest = Path('results/v3_qwen1.5b/team_lora.json')
dest.parent.mkdir(parents=True, exist_ok=True)
already = False
if dest.is_file():
    payload = json.loads(dest.read_text())
    already = payload.get('status') == 'measured' and payload.get('kind') == 'model_eval'
if already:
    print(f'{dest} is already measured. Not scoring the frozen test again.')
else:
    command = [
        sys.executable, '-m', 'orbit_agents.eval.run',
        '--split', 'test',
        '--mode', 'team',
        '--backend', 'transformers',
        '--model', 'Qwen/Qwen2.5-1.5B-Instruct',
        '--out', str(dest),
    ]
    command += ['--adapter', 'finetune/outputs/adapter']
    print(' '.join(command), flush=True)
    subprocess.check_call(command, env=env)
    save_step([str(dest)])


In [ ]:
import os, subprocess, sys
from pathlib import Path
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
folder = Path('results/v3_qwen1.5b')
needed = [folder / name for name in ('single_base.json', 'team_base.json', 'team_lora.json')]
missing = [str(path) for path in needed if not path.is_file()]
if missing:
    raise SystemExit('Missing measured files for the comparison: ' + ', '.join(missing))
command = [
    sys.executable, '-m', 'orbit_agents.eval.compare',
    '--single', str(needed[0]),
    '--team', str(needed[1]),
    '--lora', str(needed[2]),
    '--out', str(folder / 'comparison.json'),
]
print(' '.join(command), flush=True)
subprocess.check_call(command, env=env)
save_step([str(folder / 'comparison.json')])
subprocess.check_call([sys.executable, '-m', 'orbit_agents.eval.readme'], env=env)


## Optional distillation run

The cells above train `finetune/configs/qlora_medium.json` and score single base, team base, and team plus that adapter once under `results/v3_qwen1.5b/`. They skip a file that is already measured. Leave them as they are.

To train on distilled trajectories, run `python finetune/distill_teacher.py` on a machine with `ORBIT_TEACHER_API_KEY`, put `finetune/data/distill/train.jsonl` and `tune.jsonl` in the project zip, and set `ORBIT_DISTILL=1` before this cell. It trains `finetune/configs/qlora_distill.json` (`max_seq_length` 1536; a shorter cap is refused because truncation drops the assistant turn) and scores a fourth frozen run, `team_distill`. It does not replace the three runs above. If `ORBIT_DISTILL` is unset, the cell prints a skip line and does not train.

Harness v3 scores belong in `results/v3_qwen1.5b/`. The rescore cell fills any of the four that are not already measured. It does not train, and it does not score a measured file again.

Do not score the frozen test in a loop.


In [ ]:
import json, os, subprocess, sys
from pathlib import Path
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
if os.environ.get('ORBIT_DISTILL') != '1':
    print('Distill training skipped. Set ORBIT_DISTILL=1 to train finetune/configs/qlora_distill.json and score team_distill.')
else:
    train_path = Path('finetune/data/distill/train.jsonl')
    tune_path = Path('finetune/data/distill/tune.jsonl')
    if not train_path.exists() or not tune_path.exists():
        raise SystemExit('Distill JSONL is missing. Run finetune/distill_teacher.py first.')
    command = [sys.executable, 'finetune/train.py', '--config', 'finetune/configs/qlora_distill.json', '--output', 'finetune/outputs-distill']
    print(' '.join(command), flush=True)
    subprocess.check_call(command, env=env)
    save_step([
        'finetune/outputs-distill/adapter',
        'finetune/outputs-distill/loss_curve.png',
        'finetune/outputs-distill/RUN.json',
        'finetune/outputs-distill/train_log.jsonl',
    ])
    dest = Path('results/v3_qwen1.5b/team_distill.json')
    dest.parent.mkdir(parents=True, exist_ok=True)
    payload = json.loads(dest.read_text()) if dest.is_file() else {}
    if payload.get('status') == 'measured' and payload.get('kind') == 'model_eval':
        print(f'{dest} is already measured. Not scoring the frozen test again.')
    else:
        command = [
            sys.executable, '-m', 'orbit_agents.eval.run',
            '--split', 'test',
            '--mode', 'team',
            '--backend', 'transformers',
            '--model', 'Qwen/Qwen2.5-1.5B-Instruct',
            '--adapter', 'finetune/outputs-distill/adapter',
            '--out', str(dest),
        ]
        print(' '.join(command), flush=True)
        subprocess.check_call(command, env=env)
        save_step([str(dest)])
    folder = dest.parent
    needed = [folder / name for name in ('single_base.json', 'team_base.json', 'team_lora.json', 'team_distill.json')]
    if all(path.is_file() for path in needed):
        command = [
            sys.executable, '-m', 'orbit_agents.eval.compare',
            '--single', str(needed[0]),
            '--team', str(needed[1]),
            '--lora', str(needed[2]),
            '--distill', str(needed[3]),
            '--out', str(folder / 'comparison.json'),
        ]
        print(' '.join(command), flush=True)
        subprocess.check_call(command, env=env)
        save_step([str(folder / 'comparison.json')])
    subprocess.check_call([sys.executable, '-m', 'orbit_agents.eval.readme'], env=env)


## Rescore all four configs on harness v3

Set `ORBIT_RESCORE=1` before this cell. It does not train. It scores the frozen test once for any of single base, team base, team plus `finetune/outputs/adapter`, and team plus `finetune/outputs-distill/adapter` that is not already measured, then writes the comparison under `results/v3_qwen1.5b/`.

If either adapter is missing, the cell stops and does not invent a score. The v2 numbers stay archived. Do not score the frozen test in a loop.


In [ ]:
import json, os, subprocess, sys
from pathlib import Path
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
if os.environ.get('ORBIT_RESCORE') != '1':
    print('Harness v3 rescore skipped. Set ORBIT_RESCORE=1 to score single_base, team_base, team_lora, and team_distill once into results/v3_qwen1.5b/. This cell does not train.')
else:
    out = Path('results/v3_qwen1.5b')
    out.mkdir(parents=True, exist_ok=True)
    medium = Path('finetune/outputs/adapter')
    distill = Path('finetune/outputs-distill/adapter')
    if not medium.exists():
        raise SystemExit('Missing finetune/outputs/adapter. The v3 rescore does not train and does not invent a score.')
    if not distill.exists():
        raise SystemExit('Missing finetune/outputs-distill/adapter. The v3 rescore does not train and does not invent a score.')
    model = 'Qwen/Qwen2.5-1.5B-Instruct'
    runs = [
        ('single', None, out / 'single_base.json'),
        ('team', None, out / 'team_base.json'),
        ('team', str(medium), out / 'team_lora.json'),
        ('team', str(distill), out / 'team_distill.json'),
    ]
    for mode, adapter, dest in runs:
        if dest.is_file():
            payload = json.loads(dest.read_text())
            if payload.get('status') == 'measured' and payload.get('kind') == 'model_eval':
                print(f'{dest} is already measured. Not scoring the frozen test again.')
                continue
        command = [
            sys.executable, '-m', 'orbit_agents.eval.run',
            '--split', 'test',
            '--mode', mode,
            '--backend', 'transformers',
            '--model', model,
            '--out', str(dest),
        ]
        if adapter:
            command.extend(['--adapter', adapter])
        print(' '.join(command), flush=True)
        subprocess.check_call(command, env=env)
        save_step([str(dest)])
    command = [
        sys.executable, '-m', 'orbit_agents.eval.compare',
        '--single', str(out / 'single_base.json'),
        '--team', str(out / 'team_base.json'),
        '--lora', str(out / 'team_lora.json'),
        '--distill', str(out / 'team_distill.json'),
        '--out', str(out / 'comparison.json'),
    ]
    print(' '.join(command), flush=True)
    subprocess.check_call(command, env=env)
    save_step([str(out / 'comparison.json')])
    subprocess.check_call([sys.executable, '-m', 'orbit_agents.eval.readme'], env=env)


In [ ]:
import shutil
from pathlib import Path
bundle = Path('/content/orbit-outputs')
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir()
required = [
    Path('finetune/outputs/adapter'),
    Path('finetune/outputs/loss_curve.png'),
    Path('finetune/outputs/RUN.json'),
    Path('finetune/outputs/train_log.jsonl'),
    Path('results/v3_qwen1.5b/single_base.json'),
    Path('results/v3_qwen1.5b/team_base.json'),
    Path('results/v3_qwen1.5b/team_lora.json'),
    Path('results/v3_qwen1.5b/comparison.json'),
    Path('README.md'),
]
optional = [
    Path('finetune/outputs-distill/adapter'),
    Path('finetune/outputs-distill/loss_curve.png'),
    Path('finetune/outputs-distill/RUN.json'),
    Path('finetune/outputs-distill/train_log.jsonl'),
    Path('results/v3_qwen1.5b/team_distill.json'),
]
paths = list(required)
for path in optional:
    if path.exists():
        paths.append(path)
for path in paths:
    if not path.exists():
        raise SystemExit(f'missing {path}')
    target = bundle / path
    target.parent.mkdir(parents=True, exist_ok=True)
    if path.is_dir():
        shutil.copytree(path, target)
    else:
        shutil.copy2(path, target)
print('Commit only these paths:')
for path in paths:
    if path.parts[0] == 'results' or path.name == 'README.md':
        print(path)
print('Do not commit adapters or finetune/data/distill.')
print('Do not replace the pending files at the top of results/.')
archive = shutil.make_archive('/content/outputs', 'zip', bundle)
print(archive)
save_step(['/content/outputs.zip'])
try:
    from google.colab import files
    files.download('/content/outputs.zip')
except Exception as exc:
    print('Download skipped outside Colab:', type(exc).__name__)


Do not score the frozen test in a loop. Download `/content/outputs.zip` before the runtime ends. The adapter is not a task-success number; the comparison file is.
Commit `results/v3_qwen1.5b/*.json` and `README.md` only. The README results block is rewritten by `python -m orbit_agents.eval.readme`. Do not replace the pending files at the top of `results/`.
